In [1]:
import osmnx as ox



In [34]:
#create candidate locations
import numpy as np
import pandas as pd

latitudes = np.arange(21.10, 21.20, 0.01)
longitudes = np.arange(79.05, 79.15, 0.01)

locations = []

for lat in latitudes:
    for lon in longitudes:
        locations.append((lat, lon))

print("Locations:", len(locations))

Locations: 110


In [35]:
# for each location, get the nearby amenities
import osmnx as ox

place = "Nagpur, Maharashtra, India"

cafes = ox.features_from_place(
    place,
    tags={"amenity": "cafe"}
)

colleges = ox.features_from_place(
    place,
    tags={"amenity": ["college", "university"]}
)

schools = ox.features_from_place(
    place,
    tags={"amenity": "school"}
)

offices = ox.features_from_place(
    place,
    tags={"office": True}
)

In [36]:
#Step 3: Extract Features
from geopy.distance import geodesic

RADIUS = 2000 # exisitng amentites within 2km radius

def count_nearby(gdf, point):

    count = 0

    for _, row in gdf.iterrows():

        try:
            c = row.geometry.centroid

            d = geodesic(
                point,
                (c.y, c.x)
            ).meters

            if d <= RADIUS:
                count += 1

        except:
            pass

    return count

records = []

for point in locations:

    cafes_count = count_nearby(cafes, point)
    colleges_count = count_nearby(colleges, point)
    schools_count = count_nearby(schools, point)
    offices_count = count_nearby(offices, point)

    records.append({
        "Latitude": point[0],
        "Longitude": point[1],
        "Cafes": cafes_count,
        "Colleges": colleges_count,
        "Schools": schools_count,
        "Offices": offices_count
    })

df = pd.DataFrame(records)

print(df.head())

   Latitude  Longitude  Cafes  Colleges  Schools  Offices
0      21.1      79.05     11         1        5       23
1      21.1      79.06     11         1        6       27
2      21.1      79.07      5         2        6       25
3      21.1      79.08      5         1        2       22
4      21.1      79.09      0         1        3        3


In [3]:
import pandas as pd

df = pd.read_csv("D:\Business Recommendation final\Business-Recommendation\cafe_features_dataset.csv")

print(df.head())

   Latitude  Longitude  Cafes  Colleges  Schools  Offices  Cluster  \
0     21.16      79.06      2        13        4       36        1   
1     21.17      79.07      1         9        9       28        1   
2     21.15      79.06      3        10        5       28        1   
3     21.15      79.07      2         9       13       26        1   
4     21.13      79.10      1        12       28       16        3   

   Opportunity_Score  
0                160  
1                127  
2                123  
3                123  
4                122  


In [4]:
#Scale Features
from sklearn.preprocessing import StandardScaler

features = df[
    [
        "Cafes",
        "Colleges",
        "Schools",
        "Offices"
    ]
]

scaler = StandardScaler()

X = scaler.fit_transform(features)

In [32]:
#Apply K-Means
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=4,
    random_state=42
)

df["Cluster"] = kmeans.fit_predict(X)

print(df.head())

   Latitude  Longitude  Cafes  Colleges  Schools  Offices  Cluster  \
0     21.16      79.06      2        13        4       36        2   
1     21.17      79.07      1         9        9       28        2   
2     21.15      79.06      3        10        5       28        2   
3     21.15      79.07      2         9       13       26        2   
4     21.13      79.10      1        12       28       16        0   

   Opportunity_Score  
0                160  
1                127  
2                123  
3                123  
4                122  


In [33]:
from sklearn.metrics import silhouette_score

score = silhouette_score(X, df["Cluster"])

print(score)

0.5398962067849408


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


# Convert centroids back to original scale
centroids = scaler.inverse_transform(kmeans.cluster_centers_)

centroid_df = pd.DataFrame(
    centroids,
    columns=features.columns
)

print(centroid_df)

       Cafes  Colleges    Schools    Offices
0   0.157895  0.684211   2.473684   3.000000
1   2.809524  7.666667   6.904762  22.666667
2  11.700000  5.400000   9.800000  23.500000
3   0.818182  7.227273  17.409091  11.636364


In [ ]:
#Create Opportunity Score
df["Opportunity_Score"] = (
    
    4 * df["Colleges"]
    + 3 * df["Offices"]
    + 1 * df["Schools"]
    - 2 * df["Cafes"]
)

In [ ]:
#Rank Locations
df = df.sort_values(
    by="Opportunity_Score",
    ascending=False
)

print(df.head(20))

    Latitude  Longitude  Cafes  Colleges  Schools  Offices  Cluster  \
0      21.16      79.06      2        13        4       36        2   
1      21.17      79.07      1         9        9       28        2   
2      21.15      79.06      3        10        5       28        2   
3      21.15      79.07      2         9       13       26        2   
4      21.13      79.10      1        12       28       16        0   
5      21.17      79.06      3        10        3       28        2   
6      21.14      79.07      3        11       10       24        2   
7      21.16      79.07      3         9       11       26        2   
8      21.12      79.07      9         9       16       27        3   
9      21.14      79.08      1        10       13       21        2   
10     21.15      79.08      2        10       11       21        2   
11     21.17      79.05      3         6        5       28        2   
12     21.12      79.10      0        10       26       13        0   
13    

In [ ]:
#Convert Coordinates to Area Names
from geopy.geocoders import Nominatim

geolocator = Nominatim(
    user_agent="cafe_project"
)

def get_area(lat, lon):

    try:

        location = geolocator.reverse(
            (lat, lon),
            exactly_one=True
        )

        return location.address

    except:
        return "Unknown"

In [11]:
top10 = df.head(10).copy()

top10["Area"] = top10.apply(
    lambda row:
    get_area(
        row["Latitude"],
        row["Longitude"]
    ),
    axis=1
)

In [12]:
print(
    top10[
        [
            "Area",
            "Opportunity_Score",
            "Cluster"
        ]
    ]
)

                                                Area  Opportunity_Score  \
0  Seminary Hills, Nagpur City, Nagpur Urban Talu...                160   
1  Seminary Hills, Nagpur City, Nagpur Urban Talu...                127   
2  Ravi Nagar, Bharat Nagar, Nagpur City, Nagpur ...                123   
3  Dharampeth, Nagpur City, Nagpur Urban Taluka, ...                123   
4  Chandan Nagar, Mahal, Nagpur City, Nagpur Urba...                122   
5  Seminary Hills Road, Seminary Hills, Nagpur Ci...                121   
6  North Ambazari Road, Ramdaspeth, Nagpur City, ...                120   
7  Dharampeth, Nagpur City, Nagpur Urban Taluka, ...                119   
8  Dhantoli, Nagpur City, Nagpur Urban Taluka, Na...                115   
9  Dhantoli, Nagpur City, Nagpur Urban Taluka, Na...                114   

   Cluster  
0        2  
1        2  
2        2  
3        2  
4        0  
5        2  
6        2  
7        2  
8        3  
9        2  


In [ ]:
#xerox

In [ ]:
import numpy as np

latitudes = np.arange(21.10, 21.20, 0.01)
longitudes = np.arange(79.05, 79.15, 0.01)

locations = []

for lat in latitudes:
    for lon in longitudes:
        locations.append((lat, lon))

print("Total Locations:", len(locations))

In [ ]:
import osmnx as ox

place = "Nagpur, Maharashtra, India"

print("Downloading OSM data...")

schools = ox.features_from_place(
    place,
    tags={"amenity": "school"}
)

colleges = ox.features_from_place(
    place,
    tags={"amenity": ["college", "university"]}
)

offices = ox.features_from_place(
    place,
    tags={"office": True}
)

bus_stops = ox.features_from_place(
    place,
    tags={"highway": "bus_stop"}
)

xerox_shops = ox.features_from_place(
    place,
    tags={
        "shop": [
            "copyshop",
            "stationery"
        ]
    }
)

print("Download Complete")

In [ ]:
import pandas as pd
from geopy.distance import geodesic

RADIUS = 2000

def count_nearby(gdf, point):

    count = 0

    for _, row in gdf.iterrows():

        try:

            centroid = row.geometry.centroid

            distance = geodesic(
                point,
                (centroid.y, centroid.x)
            ).meters

            if distance <= RADIUS:
                count += 1

        except:
            pass

    return count

records = []

for point in locations:

    schools_count = count_nearby(
        schools,
        point
    )

    colleges_count = count_nearby(
        colleges,
        point
    )

    offices_count = count_nearby(
        offices,
        point
    )

    bus_count = count_nearby(
        bus_stops,
        point
    )

    xerox_count = count_nearby(
        xerox_shops,
        point
    )

    records.append({
        "Latitude": point[0],
        "Longitude": point[1],
        "Schools": schools_count,
        "Colleges": colleges_count,
        "Offices": offices_count,
        "BusStops": bus_count,
        "XeroxShops": xerox_count
    })

df = pd.DataFrame(records)

df.to_csv(
    "xeroxfeaturesdataset.csv",
    index=False
)

print(df.head())

In [18]:
import pandas as pd

df = pd.read_csv(r"D:\Business Recommendation final\Business-Recommendation\xeroxfeaturesdataset.csv")

print(df.head())

   Latitude  Longitude  Schools  Colleges  Offices  BusStops  XeroxShops
0      21.1      79.05        5         1       23         0           3
1      21.1      79.06        6         1       27         0           3
2      21.1      79.07        6         2       25         0           2
3      21.1      79.08        2         1       22         0           2
4      21.1      79.09        3         1        3         0           1


In [19]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

features = df[
    [
        "Schools",
        "Colleges",
        "Offices",
        "BusStops",
        "XeroxShops"
    ]
]

scaler = StandardScaler()

X = scaler.fit_transform(features)

kmeans = KMeans(
    n_clusters=4,
    random_state=42
)

df["Cluster"] = kmeans.fit_predict(X)

In [20]:
#Create Xerox Opportunity Score
df["Opportunity_Score"] = (
    5 * df["Colleges"]
    +
    4 * df["Schools"]
    +
    3 * df["Offices"]
    +
    1 * df["BusStops"]
    -
    4 * df["XeroxShops"]
)

In [21]:
#Rank Locations
df = df.sort_values(
    by="Opportunity_Score",
    ascending=False
)

print(
    df[
        [
            "Latitude",
            "Longitude",
            "Opportunity_Score"
        ]
    ].head(20)
)

    Latitude  Longitude  Opportunity_Score
38     21.13      79.10                213
67     21.16      79.06                187
28     21.12      79.11                180
27     21.12      79.10                179
57     21.15      79.07                178
39     21.13      79.11                176
24     21.12      79.07                175
26     21.12      79.09                171
46     21.14      79.07                170
68     21.16      79.07                170
47     21.14      79.08                168
79     21.17      79.07                161
58     21.15      79.08                160
37     21.13      79.09                159
23     21.12      79.06                159
56     21.15      79.06                156
35     21.13      79.07                155
29     21.12      79.12                148
12     21.11      79.06                147
36     21.13      79.08                146


In [22]:
from geopy.geocoders import Nominatim

geolocator = Nominatim(
    user_agent="xerox_recommendation"
)

def get_area(lat, lon):

    try:

        location = geolocator.reverse(
            (lat, lon),
            exactly_one=True
        )

        return location.address

    except:
        return "Unknown"

top10 = df.head(10).copy()

top10["Area"] = top10.apply(
    lambda row:
    get_area(
        row["Latitude"],
        row["Longitude"]
    ),
    axis=1
)

In [30]:
top10= top10.sort_values(
    by="Opportunity_Score",
    ascending=False
)

top10 = top10.drop_duplicates(
    subset=["Area"],
    keep="first"
).reset_index(drop=True)